# CDI Layer 2, Layer 3 and Layer 4

Use the **compute** kernel. After repository code changes, restart the kernel once. Put `OPENAI_API_KEY` in the repository `.env`, set the factsheet, plugin, requirements and Layer 2 reasoning below, and run the next cell. It creates a resumable schema-7 run: extract evidence once, choose domains, assign fact IDs and review corrections. Historical runs are read-only; Layer 3/4 integration remains separate. The cell shows only start and finish status; operational detail is written to the run log.

In [1]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2 import create_run as create_layer2_run
from ML.deep_research.layer2 import run_all as run_layer2
from ML.deep_research.layer2.backend.settings import DOMAIN_PLUGIN_PATH, RUNS_DIR
from ML.deep_research.layer2.backend.fs import load_json

FACT_SHEET_PATH = Path(r"inputs\new_fact_sheet.md")
LAYER2_DOMAIN_PLUGIN = DOMAIN_PLUGIN_PATH  # edit for another industry
LAYER2_REQUIREMENTS = Path(r"inputs\requirement.md")  # replace template placeholders before running
# Schema 6: readable results are in domains/ and domain_plan.md; not yet integrated with Layer 3.
LAYER2_REASONING_EFFORT = "high"  # low | medium | high | max

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 2.")

L2_DYNAMIC_RUN = create_layer2_run(
    FACT_SHEET_PATH,
    LAYER2_DOMAIN_PLUGIN,
    LAYER2_REQUIREMENTS,
    RUNS_DIR,
    reasoning_effort=LAYER2_REASONING_EFFORT,
)
print("Layer 2: running")
try:
    await asyncio.to_thread(run_layer2, L2_DYNAMIC_RUN)
except Exception:
    print("Layer 2: failed — see run.log")
else:
    print(f"Layer 2: {load_json(L2_DYNAMIC_RUN / 'run.json')['status']}")
print(f"Run: {L2_DYNAMIC_RUN}")
print(f"Log: {L2_DYNAMIC_RUN / 'run.log'}")


ImportError: DLL load failed while importing _uuid_utils: An Application Control policy has blocked this file.

## Layer 3 — live online research

Run this only after Layer 2 completes. **This cell confirms that the input is public or invented, sends research queries to external services, and consumes model/web-search usage.** Set the exact Layer 2 run path, Layer 3 model reasoning, web-search depth, and web-search verbosity below. A blank run path uses `L2_RUN` from the Layer 2 cell.

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2.backend.fs import load_json, read_text
from ML.deep_research.layer3.cli import run_all as run_layer3
from ML.deep_research.layer3.pipeline.create_run import create_run as create_layer3_run
from ML.deep_research.layer3.settings import RUNS_DIR as LAYER3_RUNS_DIR, SCHEMA_VERSION
from ML.deep_research.layer3.usage import summarize_usage

LAYER3_SOURCE_RUN_PATH = r"runs\inputs-new-fact-sheet-9563041d\L2_20260902_114539_0f6a"  # paste a runs/.../L2_* folder; blank uses L2_RUN above
LAYER3_MODEL_REASONING_EFFORT = "high"  # low | medium | high | max
WEB_SEARCH_DEPTH = "medium"  # low | medium | high
WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
PUBLIC_INPUT_CONFIRMED = True

load_dotenv_key()
if LAYER3_SOURCE_RUN_PATH.strip():
    L2_RUN = Path(LAYER3_SOURCE_RUN_PATH)
elif globals().get("L2_RUN"):
    L2_RUN = Path(L2_RUN)
else:
    raise RuntimeError("Set LAYER3_SOURCE_RUN_PATH to the exact Layer 2 run folder.")
print(f"Using existing Layer 2 run: {L2_RUN}")

if not PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 3 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 3.")

for run_json in sorted(
    LAYER3_RUNS_DIR.rglob("L3_*/run.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
):
    candidate = run_json.parent
    candidate_record = load_json(run_json)
    source_path = candidate_record.get("source_l2", {}).get("path", "")
    search_options = candidate_record.get("web_search", {})
    if (
        candidate_record.get("schema_version") == SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L2_RUN.resolve()
        and candidate_record.get("reasoning_effort") == LAYER3_MODEL_REASONING_EFFORT
        and search_options.get("context_size") == WEB_SEARCH_DEPTH
        and search_options.get("verbosity") == WEB_SEARCH_VERBOSITY
    ):
        L3_RUN = candidate
        break
else:
    L3_RUN = create_layer3_run(
        L2_RUN,
        LAYER3_RUNS_DIR,
        public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
        reasoning_effort=LAYER3_MODEL_REASONING_EFFORT,
        web_search_context_size=WEB_SEARCH_DEPTH,
        web_search_verbosity=WEB_SEARCH_VERBOSITY,
    )

l3_before = load_json(L3_RUN / "run.json")
execution = l3_before.get("execution", {})
resume_command = f".\\run.ps1 -ResumeL3 '{L3_RUN}'"
print(f"Layer 3 run: {L3_RUN}")
print(f"Resume if interrupted: {resume_command}")

layer3_task = None
if l3_before.get("status") != "complete":
    layer3_task = asyncio.create_task(run_layer3(L3_RUN))
while layer3_task and not layer3_task.done():
    await asyncio.sleep(5)
    live = load_json(L3_RUN / "run.json")
    domains = live.get("execution", {}).get("domains", {})
    running = [name for name, item in domains.items() if item.get("status") == "running"]
    completed = [name for name, item in domains.items() if item.get("status") == "complete"]
    domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
    lines = read_text(L3_RUN / "usage.jsonl").splitlines()
    try:
        latest = json.loads(lines[-1]) if lines else {}
    except json.JSONDecodeError:
        latest = {}
    clear_output(wait=True)
    print(f"Layer 3 run: {L3_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Active coordinator: {running[0] if running else 'none'}")
    print(f"Completed coordinators: {len(completed)}/8")
    print(f"Saved domain responses: {len(domain_finals)}/8")
    print("Usage:", summarize_usage(L3_RUN))
    if latest:
        print("Last activity:", latest.get("timestamp"), latest.get("actor"), latest.get("phase"), latest.get("detail", ""))

L3_ERROR = ""
if layer3_task:
    try:
        await layer3_task
    except Exception as error:
        L3_ERROR = f"{type(error).__name__}: {error}"
clear_output(wait=True)

l3_record = load_json(L3_RUN / "run.json")
domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
final_answer = L3_RUN / "research" / "final.md"
final_records = [*l3_record.get("execution", {}).get("domains", {}).values(), l3_record.get("execution", {}).get("final", {})]
print(f"Layer 3 run: {L3_RUN}")
print(f"Status: {l3_record.get('status', 'unknown')}")
print(f"Resume if interrupted: .\\run.ps1 -ResumeL3 '{L3_RUN}'")
if L3_ERROR:
    print(f"Execution error: {L3_ERROR}")
failed = [
    f"{item.get('stage')}: {item.get('actor')} — {item.get('error')}"
    for item in final_records
    if item.get("status") == "failed"
]
for item in failed:
    print(f"Failed stage: {item}")
display(Markdown("### Recorded Layer 3 usage"))
display(JSON(data=summarize_usage(L3_RUN), expanded=True))
print(f"Saved domain responses: {len(domain_finals)}/8")
for path in domain_finals:
    print(f"- {path.relative_to(L3_RUN)}")
display(Markdown("### Property synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; resume the incomplete Layer 3 run shown above."))


In [ ]:
# Layer 4 — live external-influence research over an existing Layer 3 run
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2.backend.fs import load_json, read_text
from ML.deep_research.layer3.usage import summarize_usage
from ML.deep_research.layer4.cli import run_all as run_layer4
from ML.deep_research.layer4.create_run import create_run as create_layer4_run
from ML.deep_research.layer4.settings import (
    DOMAIN_NAMES,
    LAYER3_HARNESS_NAME,
    LAYER3_SCHEMA_VERSION,
    SCHEMA_VERSION as LAYER4_SCHEMA_VERSION,
)

LAYER4_SOURCE_RUN_PATH = r"runs\inputs-new-fact-sheet-9563041d\L3_20260902_120424_e0de"
LAYER4_MODEL_REASONING_EFFORT = "high"  # low | medium | high | max
LAYER4_WEB_SEARCH_DEPTH = "medium"  # low | medium | high
LAYER4_WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
LAYER4_PUBLIC_INPUT_CONFIRMED = True
LAYER4_RETRY_FAILED = False

load_dotenv_key()
L3_RUN = Path(LAYER4_SOURCE_RUN_PATH).resolve()
if not (L3_RUN / "run.json").is_file():
    raise RuntimeError("Set LAYER4_SOURCE_RUN_PATH to a CDI Layer 3 run folder.")
source_l3 = load_json(L3_RUN / "run.json")
if (
    source_l3.get("schema_version") != LAYER3_SCHEMA_VERSION
    or source_l3.get("harness") != LAYER3_HARNESS_NAME
):
    raise RuntimeError("Layer 4 requires a schema-8 direct-research Layer 3 run.")
if not LAYER4_PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 4 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 4.")

L4_RUN = None
for run_json in sorted(
    L3_RUN.parent.glob("L4_*/run.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
):
    candidate = load_json(run_json)
    source_path = candidate.get("source_l3", {}).get("path", "")
    search_options = candidate.get("web_search", {})
    if (
        candidate.get("schema_version") == LAYER4_SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L3_RUN
        and candidate.get("reasoning_effort") == LAYER4_MODEL_REASONING_EFFORT
        and search_options.get("context_size") == LAYER4_WEB_SEARCH_DEPTH
        and search_options.get("verbosity") == LAYER4_WEB_SEARCH_VERBOSITY
    ):
        L4_RUN = run_json.parent
        break
if L4_RUN is None:
    L4_RUN = create_layer4_run(
        L3_RUN,
        public_input_confirmed=LAYER4_PUBLIC_INPUT_CONFIRMED,
        reasoning_effort=LAYER4_MODEL_REASONING_EFFORT,
        web_search_context_size=LAYER4_WEB_SEARCH_DEPTH,
        web_search_verbosity=LAYER4_WEB_SEARCH_VERBOSITY,
    )

l4_before = load_json(L4_RUN / "run.json")
resume_command = f".\\run.ps1 -ResumeL4 '{L4_RUN}'"
print(f"Layer 4 run: {L4_RUN}")
print(f"Resume if interrupted: {resume_command}")

layer4_task = None
if l4_before.get("status") != "complete" or LAYER4_RETRY_FAILED:
    layer4_task = asyncio.create_task(
        run_layer4(L4_RUN, retry_failed=LAYER4_RETRY_FAILED)
    )
while layer4_task and not layer4_task.done():
    await asyncio.sleep(5)
    live = load_json(L4_RUN / "run.json")
    domain_records = live.get("execution", {}).get("domains", {})
    stages = [
        (domain, stage, record)
        for domain, records in domain_records.items()
        for stage, record in records.items()
    ]
    running = [f"{domain}/{stage}" for domain, stage, record in stages if record.get("status") == "running"]
    completed = sum(record.get("status") == "complete" for _, _, record in stages)
    saved_outputs = len(list((L4_RUN / "domains").glob("*/*.md")))
    clear_output(wait=True)
    print(f"Layer 4 run: {L4_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Active stage: {running[0] if running else 'none'}")
    print(f"Completed domain stages: {completed}/24")
    print(f"Saved domain outputs: {saved_outputs}/24")
    print("Usage:", summarize_usage(L4_RUN))

L4_ERROR = ""
if layer4_task:
    try:
        await layer4_task
    except Exception as error:
        L4_ERROR = f"{type(error).__name__}: {error}"
clear_output(wait=True)

l4_record = load_json(L4_RUN / "run.json")
domain_records = l4_record.get("execution", {}).get("domains", {})
final_record = l4_record.get("execution", {}).get("final", {})
saved_outputs = sorted((L4_RUN / "domains").glob("*/*.md"))
final_answer = L4_RUN / "research" / "final.md"
print(f"Layer 4 run: {L4_RUN}")
print(f"Status: {l4_record.get('status', 'unknown')}")
print(f"Resume if interrupted: {resume_command}")
if L4_ERROR:
    print(f"Execution error: {L4_ERROR}")
for domain in DOMAIN_NAMES:
    records = domain_records.get(domain, {})
    statuses = ", ".join(
        f"{stage}={record.get('status', 'missing')}"
        for stage, record in records.items()
    )
    print(f"- {domain}: {statuses or 'missing'}")
failed = [
    record
    for records in domain_records.values()
    for record in records.values()
    if record.get("status") == "failed"
]
if final_record.get("status") == "failed":
    failed.append(final_record)
for record in failed:
    print(f"Failed stage: {record.get('stage')}/{record.get('actor')} — {record.get('error')}")
display(Markdown("### Recorded Layer 4 usage"))
display(JSON(data=summarize_usage(L4_RUN), expanded=True))
print(f"Saved domain outputs: {len(saved_outputs)}/24")
display(Markdown("### External-influence synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; resume the Layer 4 run shown above."))
